# Analyse en composantes principales

| Module | Durée | Prérequis |
|---|---|---|
| M8 – Décomposition en valeurs singulières et données | 75 min | M6 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Avant le conseil de classe de fin de trimestre, l'équipe pédagogique d'un lycée dispose des moyennes de ses 150 élèves de terminale dans 10 disciplines. Le fichier `donnees/notes_eleves.csv` les rassemble, avec les spécialités suivies par chaque élève. Les données sont simulées, mais leur structure ressemble à celle de vraies notes.

Un tableau de 1 500 nombres est illisible. La proviseure adjointe aimerait **une seule image** qui montre comment se répartissent les profils d'élèves, quelles disciplines « vont ensemble », et quels élèves ont un profil atypique.

**Les questions : peut-on résumer ces 10 notes par 2 ou 3 nombres par élève sans trop perdre d'information ? Que signifient ces nombres ?**

**Activité** : Si vous deviez décrire un élève par deux nombres seulement, lesquels choisiriez-vous ? Qu'attendez-vous des liens entre les notes de mathématiques et de physique, ou entre celles de français et de philosophie ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_part_expliquee_2_axes = 0.5      # part de l'information conservée par les 2 premiers axes (modifiez)
pari_sens_axe_1 = "sciences contre lettres"    # (modifiez)

## ③ Abstraire : l'analyse en composantes principales

Chaque élève est un point dans un espace à 10 dimensions. On cherche les directions dans lesquelles le nuage de points est le plus **étalé** : ce sont elles qui distinguent le mieux les élèves.

![Schéma : un nuage de points avec son premier axe dans la direction de plus grande dispersion et le deuxième perpendiculaire ; les axes sont les colonnes de V dans la SVD des données centrées réduites](../../assets/schemas/schema-acp.svg)

1. On **centre** chaque colonne (on retire la moyenne de la discipline) et on la **réduit** (on la divise par son écart type), pour que toutes les disciplines comptent autant.
2. On calcule la SVD de la matrice obtenue : $X = U\Sigma V^{\mathsf{T}}$.
3. Les colonnes de $V$ sont les **axes principaux** : des combinaisons des disciplines.
4. Les lignes de $U\Sigma$ donnent les **coordonnées** de chaque élève sur ces axes.
5. La part de l'information (de la dispersion totale) portée par l'axe $i$ est $\sigma_i^2 / \sum_j \sigma_j^2$.

C'est exactement l'approximation de rang faible du notebook 8.2, appliquée à des données centrées et réduites.

**Activité** : Si deux variables centrées réduites étaient exactement égales pour tous les individus, combien d'axes faudrait-il pour tout décrire ? Quel serait le premier axe ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Les données

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

notes = pd.read_csv("donnees/notes_eleves.csv")
matieres = list(notes.columns[2:])
print(notes.shape)
notes[matieres].describe().round(1)

In [ ]:
correlations = notes[matieres].corr()
fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(correlations, cmap="liouai", vmin=0, vmax=1)
ax.set_xticks(range(len(matieres)), matieres, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(matieres)), matieres, fontsize=8)
for i in range(len(matieres)):
    for j in range(len(matieres)):
        v = correlations.iloc[i, j]
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7, color="white" if v > 0.55 else LIOUAI["navy"])
fig.colorbar(im, ax=ax, shrink=0.8); ax.set_title("Corrélations entre disciplines")
plt.tight_layout(); plt.show()

**Activité** : Ce tableau de 100 nombres est-il un bon moyen de communiquer avec l'équipe ? Repérez deux blocs de disciplines fortement liées.

Indiquez vos réponses et justifiez-les.

### L'ACP par la SVD

In [ ]:
Y = notes[matieres].to_numpy()
X = (Y - Y.mean(axis=0)) / Y.std(axis=0)          # centrer et réduire
U, s, Vt = np.linalg.svd(X, full_matrices=False)
part = s ** 2 / np.sum(s ** 2)
# Orientation des signes, pour la lecture : axe 1 positif pour les maths, axe 2 positif pour les maths, axe 3 pour l'EPS
for i, repere in [(0, "mathematiques"), (1, "mathematiques"), (2, "eps")]:
    if Vt[i, matieres.index(repere)] < 0:
        U[:, i], Vt[i] = -U[:, i], -Vt[i]
coordonnees = U * s

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(range(1, 11), 100 * part, color=LIOUAI["vert"])
ax.plot(range(1, 11), 100 * np.cumsum(part), "o-", color=LIOUAI["navy"], label="cumul")
ax.set_xlabel("axe"); ax.set_ylabel("% de la dispersion"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Part de chaque axe :", ", ".join(f"{p:.1%}" for p in part[:5]))

### Que représentent les axes ? Le cercle des corrélations

Pour chaque discipline, on calcule sa corrélation avec les coordonnées des élèves sur les axes 1 et 2. Une flèche longue et proche d'un axe indique une discipline bien représentée par cet axe.

In [ ]:
correlations_axes = Vt.T * s / np.sqrt(len(X))         # corrélation de chaque discipline avec chaque axe

def cercle(i, j, ax):
    t = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(t), np.sin(t), color=LIOUAI["gris"], lw=1)
    places = []                                   # positions des étiquettes déjà posées, pour éviter les chevauchements
    for k in np.argsort(-correlations_axes[:, j]):
        m = matieres[k]
        x, y = correlations_axes[k, i], correlations_axes[k, j]
        ax.annotate("", (x, y), (0, 0), arrowprops=dict(arrowstyle="-|>", color=LIOUAI["navy"], lw=1.2))
        lx = x + 0.04 * np.sign(x)
        libre = lambda yy: not any(abs(lx - px) < 0.45 and abs(yy - py) < 0.07 for px, py in places)
        ly = next((y + d for d in (0, 0.07, -0.07, 0.14, -0.14) if libre(y + d)), y)   # près de la flèche
        places.append((lx, ly))
        ax.text(lx, ly, m, fontsize=8, ha="left" if x >= 0 else "right", va="center")
    ax.axhline(0, color=LIOUAI["gris"], lw=0.6); ax.axvline(0, color=LIOUAI["gris"], lw=0.6)
    ax.set_xlim(-1.25, 1.25); ax.set_ylim(-1.25, 1.25); ax.set_aspect("equal")
    ax.set_xlabel(f"axe {i + 1} ({part[i]:.0%})"); ax.set_ylabel(f"axe {j + 1} ({part[j]:.0%})")

fig, axes = plt.subplots(1, 2, figsize=(13, 6.2))
cercle(0, 1, axes[0]); cercle(1, 2, axes[1])
plt.tight_layout(); plt.show()

### Les élèves dans le plan des deux premiers axes

In [ ]:
specialites = notes["specialites"]
couleurs = {"maths-physique": LIOUAI["navy"], "SVT-physique": LIOUAI["vert"], "SES-HGGSP": LIOUAI["jaune"],
            "LLCER-HLP": LIOUAI["gris"], "arts-HLP": "#C0504D"}
fig, ax = plt.subplots(figsize=(8.5, 6))
for spe, couleur in couleurs.items():
    m = (specialites == spe).to_numpy()
    ax.scatter(coordonnees[m, 0], coordonnees[m, 1], s=18, color=couleur, label=spe)
remarquables = [int(np.argmax(coordonnees[:, 0])), int(np.argmin(coordonnees[:, 0])),
                int(np.argmax(coordonnees[:, 1])), int(np.argmin(coordonnees[:, 1]))]
for k in remarquables:
    ax.annotate(notes["eleve"][k], coordonnees[k, :2], textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.axhline(0, color=LIOUAI["gris"], lw=0.6); ax.axvline(0, color=LIOUAI["gris"], lw=0.6)
ax.set_xlabel(f"axe 1 ({part[0]:.0%})"); ax.set_ylabel(f"axe 2 ({part[1]:.0%})"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title("Les 150 élèves, résumés par deux nombres")
plt.tight_layout(); plt.show()

**Activité** : Retrouvez dans le tableau de notes les quatre élèves annotés. Leur position sur le graphique correspond-elle à leurs notes ?

Indiquez vos réponses et justifiez-les.

In [ ]:
for k in remarquables:
    print(notes.iloc[k][["eleve", "specialites"]].to_dict(), "  coordonnées :", coordonnees[k, :3].round(2))
    print("   ", notes.iloc[k][matieres].to_dict())

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_part_expliquee_2_axes:.0%} pour deux axes ; axe 1 = {pari_sens_axe_1}")
print(f"Résultat   : {part[:2].sum():.0%} pour deux axes, {part[:3].sum():.0%} pour trois")

**Activité** :

1. Donnez un nom à chacun des trois premiers axes, en vous appuyant sur les deux cercles des corrélations (axes 1 et 2, puis axes 2 et 3).
2. Que signifie, pour un élève, une coordonnée fortement négative sur l'axe 2 ?
3. Les spécialités se séparent-elles sur le graphique ? Selon quel axe ? Est-ce surprenant ?
4. Préparez pour le conseil de classe **une** figure et **trois** phrases. Quelle figure choisissez-vous, et pourquoi plutôt que le tableau des corrélations ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Les hypothèses.** On a réduit chaque discipline par son écart type : toutes « comptent » autant. Est-ce un choix neutre ? Que se passerait-il sans réduction, si une discipline avait des notes beaucoup plus dispersées que les autres ? (Le notebook 8.4 y revient.)
- **Interprétation prudente.** L'axe 1 mélange toutes les disciplines. Peut-on l'appeler « intelligence » ? Qu'est-ce que l'ACP ne dit pas sur les **causes** des différences de notes ?
- **Généraliser.** Écrivez une fonction `acp(tableau, reduire=True)` qui renvoie la part de chaque axe, les axes (tableau pandas avec les noms des variables) et les coordonnées des individus. Appliquez-la aux seules disciplines scientifiques, puis aux seules disciplines littéraires : combien d'axes faut-il dans chaque cas ?

In [ ]:
def acp(tableau, reduire=True):
    # À compléter : renvoie (part, axes, coordonnees)
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def acp(tableau, reduire=True):
    Y = tableau.to_numpy(dtype=float)
    X = Y - Y.mean(axis=0)
    if reduire:
        X = X / Y.std(axis=0)
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    part = s ** 2 / np.sum(s ** 2)
    axes = pd.DataFrame(Vt.T, index=tableau.columns, columns=[f"axe {i + 1}" for i in range(len(s))])
    return part, axes, U * s

for nom, groupe in [("sciences", ["mathematiques", "physique_chimie", "svt"]),
                    ("lettres", ["francais", "histoire_geo", "philosophie", "anglais", "espagnol"])]:
    part_g, axes_g, _ = acp(notes[groupe])
    print(f"{nom:9s} : parts {', '.join(f'{p:.0%}' for p in part_g)}")
    print(axes_g.iloc[:, :2].round(2).to_string(), "\n")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Énoncer le résultat dans le contexte du problème": 1,
    "Repérer les traits intéressants du résultat": 1,
    "Choisir la meilleure forme de communication": 1,
    "Identifier les concepts pertinents et leurs relations": 1,
    "Identifier, énoncer et justifier les hypothèses": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "8.3-acp", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")